# YOLO OBB Dataset Visualization

This notebook provides tools to visualize and analyze YOLO Oriented Bounding Box (OBB) datasets. It includes functions to:
- Load dataset configurations
- Display annotated images
- Analyze dataset statistics
- Visualize training and validation samples

## 1. Import Required Libraries

Import all necessary libraries for dataset processing and visualization.

In [ ]:
import os
import cv2
import yaml
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set matplotlib style for better visualizations
plt.style.use('default')
%matplotlib inline

## 2. Load Dataset Configuration

Define a function to load the dataset configuration from `data.yaml` file.

In [ ]:
def load_data_config(yaml_path):
    """Carga la configuración del dataset desde data.yaml"""
    with open(yaml_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

# Load and display configuration
config = load_data_config('data.yaml')
print("Dataset Configuration:")
print("="*50)
print(f"Path: {config.get('path', 'N/A')}")
print(f"Train: {config.get('train', 'N/A')}")
print(f"Val: {config.get('val', 'N/A')}")
print(f"\nClasses:")
for class_id, class_name in config.get('names', {}).items():
    print(f"  {class_id}: {class_name}")

## 3. Read YOLO Annotations

Implement a function to read and parse YOLO format annotations from label files.

In [ ]:
def read_yolo_annotation(label_path):
    """Lee las anotaciones en formato YOLO"""
    annotations = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    coords = list(map(float, parts[1:]))
                    annotations.append({'class_id': class_id, 'coords': coords})
    return annotations

# Test the function with a sample label file
print("Function 'read_yolo_annotation' defined successfully!")
print("This function supports both standard bounding boxes (4 coords) and OBB format (8 coords)")

## 4. Draw Annotations on Images

Create a function to draw bounding boxes or oriented bounding boxes on images with class labels.

In [ ]:
def draw_annotations(image, annotations, class_names, colors):
    """Dibuja las anotaciones sobre la imagen"""
    h, w = image.shape[:2]
    
    for ann in annotations:
        class_id = ann['class_id']
        coords = ann['coords']
        
        # Convertir coordenadas normalizadas a píxeles
        if len(coords) == 4:  # Formato xywh
            x_center, y_center, width, height = coords
            x1 = int((x_center - width/2) * w)
            y1 = int((y_center - height/2) * h)
            x2 = int((x_center + width/2) * w)
            y2 = int((y_center + height/2) * h)
            
            color = colors[class_id]
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            
            # Agregar etiqueta de clase
            class_name = class_names.get(class_id, f'Class {class_id}')
            cv2.putText(image, class_name, (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
        elif len(coords) == 8:  # Formato OBB (8 coordenadas)
            points = []
            for i in range(0, 8, 2):
                x = int(coords[i] * w)
                y = int(coords[i+1] * h)
                points.append([x, y])
            points = np.array(points, dtype=np.int32)
            color = colors[class_id]
            cv2.polylines(image, [points], True, color, 2)
            
            # Agregar etiqueta de clase
            class_name = class_names.get(class_id, f'Class {class_id}')
            cv2.putText(image, class_name, (points[0][0], points[0][1]-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return image

print("Function 'draw_annotations' defined successfully!")
print("Supports both rectangular and oriented bounding boxes")

## 5. Visualize Random Samples

Implement a function to display a grid of random sample images with their annotations.

In [ ]:
def visualize_samples(config, split='train', num_samples=6):
    """Visualiza muestras aleatorias del dataset"""
    base_path = Path(config['path'])
    images_path = base_path / config[split]
    labels_path = base_path / 'labels' / split
    
    class_names = config['names']
    
    # Generar colores únicos para cada clase
    colors = {}
    for class_id in class_names.keys():
        colors[class_id] = tuple(random.randint(0, 255) for _ in range(3))
    
    # Obtener lista de imágenes
    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png'))
    
    if len(image_files) == 0:
        print(f"No se encontraron imágenes en {images_path}")
        return
    
    # Seleccionar muestras aleatorias
    samples = random.sample(image_files, min(num_samples, len(image_files)))
    
    # Configurar la figura
    rows = (num_samples + 2) // 3
    cols = min(3, num_samples)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    
    if num_samples == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if num_samples > 3 else axes
    
    for idx, img_path in enumerate(samples):
        # Leer imagen
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Leer anotaciones
        label_path = labels_path / (img_path.stem + '.txt')
        annotations = read_yolo_annotation(label_path)
        
        # Dibujar anotaciones
        image_with_boxes = draw_annotations(image.copy(), annotations, class_names, colors)
        
        # Mostrar
        axes[idx].imshow(image_with_boxes)
        axes[idx].set_title(f'{split.upper()}: {img_path.name}\nClases: {len(annotations)}')
        axes[idx].axis('off')
    
    # Ocultar ejes vacíos
    for idx in range(len(samples), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'dataset_visualization_{split}.png', dpi=150, bbox_inches='tight')
    print(f"Visualización guardada como: dataset_visualization_{split}.png")
    plt.show()

print("Function 'visualize_samples' defined successfully!")

## 6. Calculate Dataset Statistics

Create a function to analyze the dataset and compute comprehensive statistics.

In [ ]:
def get_dataset_statistics(config):
    """Muestra estadísticas del dataset"""
    stats = {}
    base_path = Path(config['path'])
    
    for split in ['train', 'val']:
        if split not in config:
            continue
            
        images_path = base_path / config[split]
        labels_path = base_path / 'labels' / split
        
        image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png'))
        
        class_counts = {k: 0 for k in config['names'].keys()}
        total_annotations = 0
        
        for img_file in image_files:
            label_path = labels_path / (img_file.stem + '.txt')
            annotations = read_yolo_annotation(label_path)
            
            for ann in annotations:
                class_counts[ann['class_id']] += 1
                total_annotations += 1
        
        stats[split] = {
            'total_images': len(image_files),
            'total_annotations': total_annotations,
            'class_counts': class_counts
        }
    
    return stats

print("Function 'get_dataset_statistics' defined successfully!")

## 7. Display Statistics by Split

Display detailed statistics for both training and validation splits.

In [ ]:
# Calculate statistics
stats = get_dataset_statistics(config)

# Display statistics for each split
for split in ['train', 'val']:
    if split not in stats:
        continue
        
    print(f"\n{'='*50}")
    print(f"Estadísticas - {split.upper()}")
    print(f"{'='*50}")
    print(f"Total de imágenes: {stats[split]['total_images']}")
    print(f"Total de anotaciones: {stats[split]['total_annotations']}")
    print(f"\nDistribución por clase:")
    
    for class_id, class_name in config['names'].items():
        count = stats[split]['class_counts'].get(class_id, 0)
        total = stats[split]['total_annotations']
        percentage = (count / total * 100) if total > 0 else 0
        print(f"  {class_name}: {count} ({percentage:.2f}%)")

# Create a bar chart for class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for idx, split in enumerate(['train', 'val']):
    if split not in stats:
        continue
        
    class_names_list = [config['names'][k] for k in sorted(config['names'].keys())]
    counts = [stats[split]['class_counts'][k] for k in sorted(config['names'].keys())]
    
    axes[idx].bar(class_names_list, counts, color='skyblue', edgecolor='navy')
    axes[idx].set_title(f'Class Distribution - {split.upper()}')
    axes[idx].set_xlabel('Class')
    axes[idx].set_ylabel('Number of Annotations')
    axes[idx].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(counts):
        axes[idx].text(i, v + max(counts)*0.01, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
print("\nGráfico de distribución guardado como: class_distribution.png")
plt.show()

## 8. Visualize Training Samples

Execute the visualization function for training split samples.

In [ ]:
print("="*50)
print("Visualizando muestras de entrenamiento...")
print("="*50)

visualize_samples(config, split='train', num_samples=6)

## 9. Visualize Validation Samples

Execute the visualization function for validation split samples.

In [ ]:
print("="*50)
print("Visualizando muestras de validación...")
print("="*50)

visualize_samples(config, split='val', num_samples=6)

## Summary

This notebook has successfully:
- ✅ Loaded dataset configuration from `data.yaml`
- ✅ Implemented functions to read YOLO annotations (both standard and OBB format)
- ✅ Created visualization tools for annotated images
- ✅ Analyzed dataset statistics and class distributions
- ✅ Generated visualizations for both training and validation samples

All visualization outputs have been saved as PNG files in the current directory.